# Validate Routing Outputs

Run lightweight checks on generated routing features, sparse SLX edge lists, and yearly POI files.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
ROUTING_DATA = PROJECT_DIR / "ANAL" / "data" / "routing"
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"
YEARS = range(2015, 2026)

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
REPORT_ROOT = ROUTING_DATA / "reports"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
active_grid_ids = set(active_cells["grid_id"])


def check(condition: bool, message: str) -> dict:
    return {"check": message, "status": "OK" if condition else "CHECK"}


def validate_pois(year: int) -> list[dict]:
    path = OSM_DIR / f"austria-{year}-pois.geoparquet"
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    pois = gpd.read_parquet(path)
    expected_types = {"motorway_exit", "rail_station", "regional_centre", "urban_centre", "higher_education"}
    return [
        check(len(pois) > 0, "POI file has rows"),
        check("poi_type" in pois.columns, "POI file has poi_type"),
        check(set(pois["poi_type"]).issubset(expected_types), "POI types are expected"),
        check(expected_types.issubset(set(pois["poi_type"])), "all routed POI types are present"),
        check("static_destination" in pois.columns, "POI file has static_destination flag"),
        check(pois.geometry.notna().all(), "POI geometries are present"),
    ]


def validate_slx_edges(path: Path) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    edges = pd.read_parquet(path)
    row_sums = edges.groupby("origin_grid_id")["weight_rowstd"].sum()
    return [
        check((edges["origin_grid_id"] != edges["destination_grid_id"]).all(), "no self-neighbors"),
        check((edges["network_distance_m"] > 0).all(), "network distances are positive"),
        check((edges["network_distance_m"] <= 1000).all(), "main SLX cutoff is respected"),
        check((edges["weight_raw"] > 0).all(), "raw weights are positive"),
        check(np.allclose(row_sums, 1.0, atol=1e-6), "row-standardized weights sum to 1"),
        check(set(edges["origin_grid_id"]).issubset(active_grid_ids), "origin IDs exist in active cells"),
        check(set(edges["destination_grid_id"]).issubset(active_grid_ids), "destination IDs exist in active cells"),
    ]

In [ ]:
records = []
for year in YEARS:
    for result in validate_pois(year):
        records.append({"year": year, "product": "pois", **result})
    slx_path = ROUTING_DATA / "matrices" / str(year) / "W_local_drive_1km_hl500m_edges.parquet"
    if slx_path.exists():
        for result in validate_slx_edges(slx_path):
            records.append({"year": year, "product": "slx_edges", **result})

validation = pd.DataFrame(records)
validation.to_csv(REPORT_ROOT / "routing_validation_summary.csv", index=False)
validation